In [ ]:
# ---- n14-e128-refits: scripts/local_seeds.py jobs on Kaggle T4 x2, one job list per GPU, run in parallel ----
# Saner Turhaner's scorer kit (sanpier/casmi-spectrum-model-v1, unchanged train.py); cache from casmi26-n0-scorer-prep.
import json, os, subprocess, sys, time
from pathlib import Path
W = Path('/kaggle/working'); INP = Path('/kaggle/input'); T0 = time.time()
def log(*a): print(f'[{time.time() - T0:7.0f}s]', *a, flush=True)
JOBS = {"0": ["refit 229 231 --epochs 128"], "1": ["refit 230 232 --epochs 128"]}
KIT = next(p.parent for p in INP.rglob('train.py') if (p.parent / 'final_model.pt').is_file() and (p.parent / 'model.py').is_file())
CACHE = next(p.parent for p in INP.rglob('peaks.bin') if (p.parent / 'fingerprints.npy').is_file())
OUT = W / 'seeds_out'; OUT.mkdir(exist_ok=True)
SRC = "\"\"\"Train and evaluate seeds of Saner Turhaner's spectrum-structure scorer on the workstation GPU (RTX 2070).\n\nSame recipe as the Kaggle kernels n1/n2/n3 (kit `sanpier/casmi-spectrum-model-v1`, cache from n0-scorer-prep):\n\n    .venv-gpu/Scripts/python.exe scripts/local_seeds.py holdout 3 4 5      # train-split runs -> holdout/casmi26_holdout_<k>.pt\n    .venv-gpu/Scripts/python.exe scripts/local_seeds.py refit 17 18        # full-data refits -> refit/casmi26_seed_<k>.pt\n    .venv-gpu/Scripts/python.exe scripts/local_seeds.py diag               # ensemble-size curve on the kit's 1000-molecule holdout\n    ... holdout 1 2 3 --split np / diag --split np                         # same on held-out Enveda natural products\n    ... holdout 1 --split np --epochs 32                                   # epoch-count check (history has every epoch)\n\nSeed k uses config seed 20261023 + 1000*k (the kit's own model is k = 0). Holdout models are the kit's\nbest_validation_model.pt, n1's holdout seeds 1-2 (outputs/local/n1/seeds) and every local holdout run.\nCPU use is kept low (2 torch threads); the data arrays are preloaded into RAM once per process.\n\"\"\"\nimport argparse\nimport json\nimport math\nimport os\nimport random\nimport re\nimport shutil\nimport sys\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\n\nROOT = Path(__file__).resolve().parents[1]\n# CASMI_KIT / CASMI_CACHE / CASMI_OUT override the paths (Kaggle jobs from scripts/build_gpu_job.py set them).\nKIT = Path(os.environ.get(\"CASMI_KIT\", ROOT / \"outputs/local/kit\"))\nCACHE = Path(os.environ.get(\"CASMI_CACHE\", ROOT / \"outputs/local/n0/cache\"))\nOUT = Path(os.environ.get(\"CASMI_OUT\", ROOT / \"outputs/local/seeds\"))\nN1_HOLDOUT = ROOT / \"outputs/local/n1/seeds\"\nBASE_SEED = 20261023\nT0 = time.time()\n# Config overrides from --set key=value (e.g. width=256 layers=4); they become part of every checkpoint/report name.\nOVERRIDES = {}\n\n\ndef log(*a):\n    print(f\"[{time.time() - T0:7.0f}s]\", *a, flush=True)\n\n\nsys.path.insert(0, str(KIT))\nimport train as T  # noqa: E402\nfrom model import SpectrumCandidateModel  # noqa: E402\n\n# hard_pools is deterministic in (allowed ids, count); build each distinct pool set once per process.\n_pool_cache, _hard_pools = {}, T.hard_pools\n\n\ndef cached_pools(mol, allowed, count):\n    a = np.asarray(allowed)\n    key = (len(a), int(a.sum()), int(a[0]), count)\n    if key not in _pool_cache:\n        t = time.time()\n        _pool_cache[key] = _hard_pools(mol, a, count)\n        log(\"POOLS\", key, round(time.time() - t), \"s\")\n    return _pool_cache[key]\n\n\nT.hard_pools = cached_pools\n\n\ndef setup(split=\"kit\"):\n    assert torch.cuda.is_available(), \"CUDA GPU required\"\n    torch.set_num_threads(2)\n    cfg = json.loads((KIT / \"config.json\").read_text())\n    cfg.update(OVERRIDES)\n    cfg[\"work\"] = str(CACHE)\n    data = T.Data(cfg)\n    if split == \"np\":\n        # Test-like holdout: the Enveda natural-product molecules listed in np_holdout_keys.txt (built from x6) leave\n        # training entirely; the kit's random validation molecules go back into training.\n        keys = set((OUT / \"np_holdout_keys.txt\").read_text().split())\n        is_val = data.mol.key.isin(keys).values\n        data.mol[\"split\"] = np.where(is_val, \"validation\", \"train\")\n        data.train, data.val = np.where(~is_val)[0], np.where(is_val)[0]\n        log(\"NP SPLIT\", len(keys), \"keys listed,\", len(data.val), \"held out\")\n    t = time.time()\n    data.x = np.array(data.x)\n    data.meta = np.array(data.meta)\n    data.fp = np.array(data.fp)\n    log(\"PRELOADED\", round(time.time() - t), \"s;\", len(data.mol), \"molecules\", len(data.spec), \"spectra\",\n        len(data.train), \"train\", len(data.val), \"validation\")\n    return cfg, data, torch.device(\"cuda\")\n\n\ndef load(cfg, path, device):\n    m = SpectrumCandidateModel(cfg).to(device)\n    m.load_state_dict(torch.load(path, map_location=device, weights_only=True)[\"state_dict\"])\n    return m.eval()\n\n\ndef candidates(cfg, data, split):\n    path = OUT / f\"validation_candidates_{split}.npz\"\n    if path.is_file():\n        z = np.load(path)\n        return {int(k): z[k] for k in z.files}\n    c = T.candidate_sets(data, cfg)\n    OUT.mkdir(parents=True, exist_ok=True)\n    np.savez_compressed(path, **{str(i): v for i, v in c.items()})\n    return c\n\n\n@torch.inference_mode()\ndef model_scores(cfg, data, m, cands, device):\n    \"\"\"{molecule: (cos term, fp term)} for one model over every candidate set (gold at index 0).\"\"\"\n    out = {}\n    for i, ids in cands.items():\n        sr = data.by_mol[i]\n        sr = sr[data.spec.source.values[sr] == 0]\n        if not len(sr):\n            continue\n        zsum = 0\n        psum = 0\n        n = 0\n        for start in range(0, len(sr), cfg[\"batch_size\"]):\n            batch = sr[start:start + cfg[\"batch_size\"]]\n            z, p = m.encode_spectra(*data.batch(batch, device))\n            zsum = zsum + z.sum(0)\n            psum = psum + p.sum(0)\n            n += len(batch)\n        fp = torch.tensor(np.asarray(data.fp[ids]), device=device, dtype=torch.float32)\n        cos = (m.encode_molecules(fp) @ (zsum / n)) / cfg[\"temperature\"]\n        fpt = cfg[\"fp_score_weight\"] * (fp @ (psum / n)) / cfg[\"fp_bits\"]\n        out[i] = (cos.float().cpu().numpy(), fpt.float().cpu().numpy())\n    return out\n\n\ndef metrics(score_lists):\n    ranks = np.array([1 + int(np.sum(s[1:] >= s[0])) for s in score_lists])\n    return dict(mrr25=float(np.mean(np.where(ranks <= 25, 1 / ranks, 0.))), top1=float(np.mean(ranks == 1)),\n                top5=float(np.mean(ranks <= 5)), n=len(ranks))\n\n\ndef ensemble(per_model, keys, mode=\"raw\"):\n    \"\"\"Average the models' total scores (raw: what the submission's CKPT_ENS module does).\"\"\"\n    lists = []\n    for i in keys:\n        parts = [pm[i][0] + pm[i][1] for pm in per_model]\n        if mode == \"z\":\n            parts = [(s - s.mean()) / (s.std() + 1e-9) for s in parts]\n        lists.append(np.mean(parts, axis=0))\n    return metrics(lists)\n\n\ndef tag_of(epochs, cosine, lr=None):\n    \"\"\"Checkpoint/report suffix: '' for the kit recipe (16 epochs, constant LR 2e-4), else\n    _e<epochs>[c][_lr<lr>][_<key><value>-...] (the last part from --set overrides).\"\"\"\n    if epochs == 16 and not cosine and lr is None and not OVERRIDES:\n        return \"\"\n    return (f\"_e{epochs}\" + (\"c\" if cosine else \"\") + (f\"_lr{lr:g}\" if lr is not None else \"\")\n            + (\"_\" + \"-\".join(f\"{k}{v}\" for k, v in sorted(OVERRIDES.items())) if OVERRIDES else \"\"))\n\n\ndef use_cosine(total_steps):\n    \"\"\"Swap torch.optim.AdamW (which train_stage builds) for one whose LR follows a cosine from lr to 0.\"\"\"\n    base = torch.optim.AdamW\n\n    class CosineAdamW(base):\n        def __init__(self, params, lr, **kw):\n            super().__init__(params, lr=lr, **kw)\n            self._lr0, self._t = lr, 0\n\n        def step(self, closure=None):\n            self._t += 1\n            f = 0.5 * (1 + math.cos(math.pi * min(self._t, total_steps) / total_steps))\n            for g in self.param_groups:\n                g[\"lr\"] = self._lr0 * f\n            return super().step(closure)\n\n    torch.optim.AdamW = CosineAdamW\n    return base\n\n\ndef train(kind, seeds, epochs, split, cosine=False, lr=None):\n    cfg, data, device = setup(split)\n    tag = kind if split == \"kit\" else f\"{kind}_{split}\"\n    folder = OUT / tag\n    folder.mkdir(parents=True, exist_ok=True)\n    report_path = OUT / f\"{tag}_report.json\"\n    report = json.loads(report_path.read_text()) if report_path.is_file() else {}\n    for k in seeds:\n        name = f\"casmi26_holdout_{k}.pt\" if kind == \"holdout\" else f\"casmi26_seed_{k}.pt\"\n        name = name.replace(\".pt\", tag_of(epochs, cosine, lr) + \".pt\")\n        if (folder / name).is_file():\n            log(\"SKIP\", name, \"exists\")\n            continue\n        run = OUT / \"runs\" / f\"{tag}_{k}\"\n        shutil.rmtree(run, ignore_errors=True)\n        run.mkdir(parents=True)\n        # Fixed recipe like the refits: no early stopping, keep the last epoch. The kit's own holdout run keeps the\n        # best epoch on its validation set, which would select on the very molecules the diagnostic scores.\n        c = dict(cfg, seed=BASE_SEED + 1000 * k, patience=10 ** 6, **({\"lr\": lr} if lr is not None else {}))\n        plain = use_cosine(epochs * cfg[\"steps_per_epoch\"]) if cosine else None\n        t = time.time()\n        if kind == \"holdout\":\n            res = T.train_stage(data, c, run, device, epochs=epochs, deadline=time.monotonic() + 3 * 3600)\n            shutil.move(str(run / \"last_validation_model.pt\"), str(folder / name))\n            hist = json.loads((run / \"training_history.json\").read_text())\n        else:\n            res = T.train_stage(data, c, run, device, refit=True, epochs=epochs, deadline=time.monotonic() + 3 * 3600)\n            shutil.move(str(run / \"final_model.pt\"), str(folder / name))\n            hist = json.loads((run / \"refit_history.json\").read_text())\n        if OVERRIDES:\n            # The pipeline's load_model builds the model from the kit's config.json; a patched load_model applies these.\n            ck = torch.load(folder / name, map_location=\"cpu\", weights_only=True)\n            ck[\"cfg_overrides\"] = dict(OVERRIDES)\n            torch.save(ck, folder / name)\n        if plain is not None:\n            torch.optim.AdamW = plain\n        report[name] = dict(res, seconds=round(time.time() - t), seed=c[\"seed\"], epochs=epochs, cosine=cosine, lr=c[\"lr\"], history=hist)\n        report_path.write_text(json.dumps(report, indent=2, default=str))\n        shutil.rmtree(run, ignore_errors=True)\n        log(\"DONE\", tag, name, report[name][\"seconds\"], \"s\", res)\n\n\ndef holdout_models(split, epochs=16, cosine=False, lr=None):\n    \"\"\"(name, path) of the holdout models trained on this split for `epochs` epochs, kit/seed order.\"\"\"\n    suffix = tag_of(epochs, cosine, lr)\n    def seeds(*dirs):\n        return {int(m.group(1)): p for d in dirs for p in d.glob(\"casmi26_holdout_*.pt\")\n                if (m := re.fullmatch(rf\"casmi26_holdout_(\\d+){suffix}\\.pt\", p.name))}\n    if split == \"np\":\n        found = seeds(OUT / \"holdout_np\")\n        return [(f\"np{k}{suffix}\", found[k]) for k in sorted(found)]\n    found = seeds(N1_HOLDOUT, OUT / \"holdout\")\n    kit = [(\"kit\", KIT / \"best_validation_model.pt\")] if epochs == 16 else []\n    return kit + [(f\"{k}{suffix}\", found[k]) for k in sorted(found)]\n\n\ndef diag(orders, split, epochs=16, cosine=False, lr=None):\n    cfg, data, device = setup(split)\n    cands = candidates(cfg, data, split)\n    log(\"CANDIDATES\", len(cands))\n    cache_dir = OUT / \"scores\"\n    cache_dir.mkdir(parents=True, exist_ok=True)\n    names, per_model = [], []\n    for name, path in holdout_models(split, epochs, cosine, lr):\n        f = cache_dir / f\"{split}_{name}_{path.stat().st_size}_{int(path.stat().st_mtime)}.npz\"\n        if f.is_file():\n            z = np.load(f)\n            s = {int(k[2:]): (z[k], z[\"f_\" + k[2:]]) for k in z.files if k.startswith(\"c_\")}\n        else:\n            s = model_scores(cfg, data, load(cfg, path, device), cands, device)\n            np.savez_compressed(f, **{f\"c_{i}\": v[0] for i, v in s.items()}, **{f\"f_{i}\": v[1] for i, v in s.items()})\n        names.append(name)\n        per_model.append(s)\n        log(\"SCORED\", name, json.dumps(metrics([s[i][0] + s[i][1] for i in s])))\n    keys = sorted(per_model[0])\n    result = dict(models=names, single={n: ensemble([pm], keys) for n, pm in zip(names, per_model)}, curve={}, z={})\n    rng = random.Random(0)\n    for n in range(2, len(per_model) + 1):\n        # Mean over random subsets of size n (the first subset is the models in listed order).\n        subsets = [list(range(n))] + [rng.sample(range(len(per_model)), n) for _ in range(orders - 1)]\n        if n == len(per_model):\n            subsets = subsets[:1]\n        rows = [ensemble([per_model[j] for j in s], keys) for s in subsets]\n        result[\"curve\"][n] = {k: float(np.mean([r[k] for r in rows])) for k in (\"mrr25\", \"top1\", \"top5\")}\n        result[\"z\"][n] = ensemble([per_model[j] for j in subsets[0]], keys, mode=\"z\")\n        log(\"ENSEMBLE\", n, json.dumps(result[\"curve\"][n]))\n    tag = split + tag_of(epochs, cosine, lr)\n    (OUT / f\"diag_{tag}_report.json\").write_text(json.dumps(result, indent=2))\n    log(\"DIAG written\", OUT / f\"diag_{tag}_report.json\")\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\"mode\", choices=[\"holdout\", \"refit\", \"diag\"])\n    ap.add_argument(\"seeds\", nargs=\"*\", type=int)\n    ap.add_argument(\"--epochs\", type=int, default=16)\n    ap.add_argument(\"--cosine\", action=\"store_true\", help=\"cosine LR decay to 0 over the run (kit: constant LR)\")\n    ap.add_argument(\"--lr\", type=float, default=None, help=\"learning rate (kit: 2e-4)\")\n    ap.add_argument(\"--split\", choices=[\"kit\", \"np\"], default=\"kit\",\n                    help=\"kit: the kit's random 10%% holdout; np: Enveda natural products held out (np_holdout_keys.txt)\")\n    ap.add_argument(\"--orders\", type=int, default=5, help=\"random subsets per ensemble size in diag\")\n    ap.add_argument(\"--set\", action=\"append\", default=[], metavar=\"KEY=VALUE\",\n                    help=\"override a kit config value (int/float), e.g. --set width=256 --set layers=4\")\n    a = ap.parse_args()\n    for kv in a.set:\n        k, v = kv.split(\"=\", 1)\n        kit = json.loads((KIT / \"config.json\").read_text())\n        assert k in kit, f\"unknown config key {k}\"\n        OVERRIDES[k] = type(kit[k])(float(v)) if isinstance(kit[k], int) else type(kit[k])(v)\n    if a.mode == \"diag\":\n        diag(a.orders, a.split, a.epochs, a.cosine, a.lr)\n    else:\n        assert a.seeds and 0 not in a.seeds, \"seed 0 is the kit's own model\"\n        assert a.mode == \"holdout\" or a.split == \"kit\", \"refits use all molecules\"\n        train(a.mode, a.seeds, a.epochs, a.split, a.cosine, a.lr)\n\n\nif __name__ == \"__main__\":\n    main()\n"
NP_KEYS = "AAWZDTNXLSGCEK\nACNHBCIZLNNLRS\nACWBQPMHZXGDFX\nADEBPBSSDYVVLD\nAGOYDEPGAOXOCK\nAIONOLUJZLIMTK\nAMOGMTLMADGEOQ\nANJTVLIZGCUXLD\nAPAJFZPFBHMFQR\nAQCHWTWZEMGIFD\nASUTZQLVASHGKV\nATBZZQPALSPNMF\nAUVVAXYIELKVAI\nBBMULGJBVDDDNI\nBDCDNTVZSILEOY\nBHQCQFFYRZLCQQ\nBJRNKVDFDLYUGJ\nBLGXFZZNTVWLAY\nBMLIIPOXVWESJG\nBXNJHAXVSOCGBA\nCEUORZQYGODEFX\nCGIGDMFJXJATDK\nCJDRUOGAGYHKKD\nCJHYKSSBQRABTM\nCKEXCBVNKRHAMX\nCKLJMWTZIZZHCS\nCLEXYFLHGFJONT\nCMKFQVZJOWHHDV\nCRSFLLTWRCYNNX\nCTRLABGOLIVAIY\nCWVRJTMFETXNAD\nCZMRCDWAGMRECN\nDATAGRPVKZEWHA\nDBEPLOCGEIEOCV\nDFMAXQKDIGCMTL\nDOFJNFPSMUCECH\nDOUMFZQKYFQNTF\nDUXQKCCELUKXOE\nDVGGLGXQSFURLP\nDWTTZBARDOXEAM\nDXDRHHKMWQZJHT\nFBPFZTCFMRRESA\nFBWWXAGANVJTLU\nFELGMEQIXOGIFQ\nFFGPTBGBLSHEPO\nFJKROLUGYXJWQN\nFSNCEEGOMTYXKY\nFTVWIRXFELQLPI\nFVYXIJYOAGAUQK\nFXNFHKRTJBSTCS\nGBOGMAARMMDZGR\nGBXSMTUPTTWBMN\nGFNANZIMVAIWHM\nGGJRAQULURVTAJ\nGGXICVAJURFBLW\nGMBQZIIUCVWOCD\nGRBXNADBNJGZRK\nGYSCAQFHASJXRS\nGZSOSUNBTXMUFQ\nGZYPWOGIYAIIPV\nHDACQVRGBOVJII\nHDOVUKNUBWVHOX\nHELXLJCILKEWJH\nHEYZWPRKKUGDCR\nHGJXAVROWQLCTP\nHHJUWIANJFBDHT\nHPNSFSBZBAHARI\nHSHNITRMYYLLCV\nHSMPDPBYAYSOBC\nHUSISCNTLUEZCN\nHYBRYAPKQCZIAE\nHYFHYPWGAURHIV\nIECPWNUMDGFDKC\nIEDVJHCEMCRBQM\nIFIQVSCCFRXSJV\nIKGXIBQEEMLURG\nIKIIZLYTISPENI\nIKMDFBPHZNJCSN\nILEDWLMCKZNDJK\nIOUVKUPGCMBWBT\nIPVQLZZIHOAWMC\nIQHUITKNHOKGFC\nIVOMOUWHDPKRLL\nIYGYMKDQCDOMRE\nJDUWHZOLEDOQSR\nJUVIOZPCNVVQFO\nJXLYSJRDGCGARV\nJXSVIVRDWWRQRT\nJYGXADMDTFJGBT\nJYXSWDCPHRTYGU\nKBDSLGBFQAGHBE\nKKRXDNYRUZGPFM\nKLLGGGQNRTVBSU\nKPYSYYIEGFHWSV\nKQNZDYYTLMIZCT\nKSQNKZMAMGACTL\nKTGFOCFYOZQVRJ\nKTZNZCYTXQYEHT\nKWBPKUMWVXUSCA\nKYWSCMDFVARMPN\nLESXFEZIFXFIQR\nLLQCRTZROWMVOL\nLOIYMIARKYCTBW\nLPMXVESGRSUGHW\nLSQZJLSUYDQPKJ\nLVVKXRQZSRUVPY\nLXNHXLLTXMVWPM\nLZFZMUMEGBBDTC\nMALFODICFSIXPO\nMATGKVZWFZHCLI\nMBNGWHIJMBWFHU\nMBRLOUHOWLUMFF\nMFWNKCLOYSRHCJ\nMGJLSBDCWOSMHL\nMKUXAQIIEYXACX\nMXYUKLILVYORSK\nNGSWKAQJJWESNS\nNODILNFGTFIURN\nNTBUBTCXACOEEC\nNWBJYWHLCVSVIJ\nNYHBQMYGNKIUIF\nOGWKCGZFUXNPDA\nOIGNJSKKLXVSLS\nOQIQSTLJSLGHID\nORJVQPIHKOARKV\nOZWKMVRBQXNZKK\nPCZOHLXUXFIOCF\nPFTAWBLQPZVEMU\nPHLBKPHSAVXXEF\nPLAPMLGJVGLZOV\nPNVNVHUZROJLTJ\nPRAUVHZJPXOEIF\nPSIFNNKUMBGKDQ\nPZPPOCZWRGNKIR\nQAIPRVGONGVQAS\nQANMHLXAZMSUEX\nQCHFTSOMWOSFHM\nQEVHRUUCFGRFIF\nQGXBDMJGAMFCBF\nQIVBCDIJIAJPQS\nQJPCKAJTLHDNCS\nQKXGRPHIBLICSC\nQPCYNIYZPDJCMB\nQQTZGKRXYRFKIF\nQXKHYNVANLEOEG\nQYIYFLOTGYLRGG\nRCFGIEPQSDGMJJ\nRFDAIACWWDREDC\nRGNXWPVNPFAADO\nRHMXXJGYXNZAPX\nRKUNBYITZUJHSG\nRMMXLENWKUUMAY\nRNKMOGIPOMVCHO\nRODXRVNMMDRFIK\nRONZAEMNMFQXRA\nRTIXKCRFFJGDFG\nRUDATBOHQWOJDD\nRWQKHEORZBHNRI\nRXGUTQNKCXHALN\nRXPRRQLKFXBCSJ\nRYMZZMVNJRMUDD\nRYYVLZVUVIJVGH\nRZGFUGXQKMEMOO\nRZJQGNCSTQAWON\nSDIVYZXRQHWCKF\nSEEGHKWOBVVBTQ\nSGEWCQFRYRRZDC\nSHIBSTMRCDJXLN\nSLYDIPAXCVVRNY\nSPJFYYJXNPEZDW\nSQFSKOYWJBQGKQ\nSQOJOAFXDQDRGF\nSTECJAGHUSJQJN\nSTZYTFJPGGDRJD\nSUBDBMMJDZJVOS\nTWDKBDSVUUKABK\nTZBJGXHYKVUXJN\nTZJALUIVHRYQQB\nUEDUENGHJMELGK\nUIDGLYUNOUKLBM\nUIKROCXWUNQSPJ\nULGZDMOVFRHVEP\nUPALKCDWIDYZDP\nURFCJEUYXNAHFI\nUYRHHBXYXSYGHA\nUZKQTCBAMSWPJD\nVAOCPAMSLUNLGC\nVGEREEWJJVICBM\nVHBFFQKBGNRLFZ\nVNONINPVFQTJOC\nVRSRXLJTYQVOHC\nWCGUUGGRBIKTOS\nWIONIXOBNMDJFJ\nWJXSWCUQABXPFS\nWMBWREPUVVBILR\nWOXKDUGGOYFFRN\nWQMLFJWIKARBFW\nWSQWAMGRHJQANC\nWTDRDQBEARUVNC\nWYJAPUKIYAZSEM\nXEVQXKKKAVVSMW\nXFZJEEAOWLFHDH\nXGYIMTFOTBMPFP\nXHCADAYNFIFUHF\nXJLATMLVMSFZBN\nXJQPQKLURWNAAH\nXKLMZUWKNUAPSZ\nXNXXZRQPTAQILV\nXOJVHLIYNSOZOO\nXQCGNURMLWFQJR\nXQYZDYMELSJDRZ\nXRWQBDJPMXRDOQ\nXUCIJNAGGSZNQT\nXUFQPHANEAPEMJ\nXVPBINOPNYFXID\nXYYAFLHHHZVPRN\nXZTUSOXSLKTKJQ\nYGPSJZOEDVAXAB\nYKJYKKNCCRKFSL\nYKPUWZUDDOIDPM\nYKRGDOXKVOZESV\nYRRAGUMVDQQZIY\nYWJXCIXBAKGUKZ\nYZMKFMIZNSOPSN\nZBHUUXLHDOUMKM\nZFXYFBGIUFBOJW\nZGLXUQQMLLIKAN\nZHSOTLOTTDYIIK\nZJTJUVIJVLLGSP\nZKLPARSLTMPFCP\nZNJFBWYDHIGLCU\nZRJBHWIHUMBLCN\nZTHLHHDJRXJGRX\nZZVUWRFHKOJYTH"
(W / 'local_seeds.py').write_text(SRC); (OUT / 'np_holdout_keys.txt').write_text(NP_KEYS)
import torch
log('GPUS', torch.cuda.device_count(), [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())], 'CPUS', os.cpu_count())
procs = {}
for g, jobs in JOBS.items():
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(g), CASMI_KIT=str(KIT), CASMI_CACHE=str(CACHE), CASMI_OUT=str(OUT))
    chain = ' ; '.join(f'{sys.executable} -u {W / "local_seeds.py"} {j}' for j in jobs)
    procs[g] = subprocess.Popen(['bash', '-c', chain], env=env, stdout=open(W / f'gpu{g}.log', 'w'), stderr=subprocess.STDOUT)
    log('GPU', g, chain)
while any(p.poll() is None for p in procs.values()):
    time.sleep(60)
    for g in procs:
        lines = [l for l in (W / f'gpu{g}.log').read_text().splitlines() if not l.startswith(('HOLDOUT {', 'REFIT {'))]
        log('GPU', g, 'rc', procs[g].poll(), '|', lines[-1][:200] if lines else '')
for g in procs:
    print(f'===== gpu{g}.log (rc {procs[g].returncode})'); print((W / f'gpu{g}.log').read_text()[-6000:])
for f in sorted(OUT.rglob('*_report.json')):
    for k, v in json.loads(f.read_text()).items():
        print('REPORT', f.name, k, v.get('epochs'), v.get('seconds'), [round(h['diagnostic_mrr25'], 3) for h in v.get('history', []) if 'diagnostic_mrr25' in h])
log('JOB_DONE', sorted(str(p.relative_to(OUT)) for p in OUT.rglob('*.pt')))
